# Phase 4: SFT — Fine-Tune Qwen3 for Tool Calling

Supervised fine-tuning on the teacher trajectories from Phase 2.
Teaches the model: which tools to call, what arguments to pass, when to stop, and how to write the final snapshot.

**Two paths:**
- **MLX (Mac)** — `mlx_lm.lora` on Apple Silicon. Uses Qwen3.5-4B-MLX-4bit. Good for testing the pipeline locally.
- **Databricks (GPU)** — Unsloth + SFTTrainer on CUDA. Uses Qwen3-4B. Full-scale training.

Both use the same training data (`trajectories_sft.jsonl`) and produce models evaluated with the same agent loop.

## MLX SFT (Mac)

Train locally on Apple Silicon using `mlx_lm.lora`. This is the same LoRA approach as Unsloth
but runs on Metal instead of CUDA.

**Key differences from Databricks path:**

| | MLX (Mac) | Unsloth (Databricks) |
|---|---|---|
| Base model | `mlx-community/Qwen3.5-4B-MLX-4bit` | `unsloth/Qwen3-4B` |
| LoRA config | YAML: `rank: 32, scale: 2.0` | Python: `r=32, lora_alpha=64` |
| Masking | `--mask-prompt` flag | `train_on_responses_only()` |
| Batch size | 1 (16GB memory) | 4 (GPU VRAM) |
| Sequence length | 8192 (with `grad_checkpoint`) | 8192 |

**Important:**
- MLX uses `scale = alpha / rank`. So `rank=32, scale=2.0` is equivalent to Unsloth's `r=32, lora_alpha=64`.
- Qwen3.5's Jinja template expects tool call `arguments` as a dict, not a JSON string. The data prep cell handles this conversion.
- `grad_checkpoint=true` is required on 16GB Mac to fit 8192 sequence length.

In [1]:
import json
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS
from bbb.helpers__data_gen import SYSTEM_PROMPT, TICKERS, make_user_prompt
from bbb.helpers__inference import compute_reward
from bbb.agent import run_tool_calling_agent_chat, TOOL_SCHEMAS_CHAT

VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

### Data Prep

MLX expects `train.jsonl` and `valid.jsonl` in a data directory. Each line is a JSON object
with `messages` and `tools` keys — exactly what `trajectories_sft.jsonl` already contains.

We split 85/15 and save to `data/bbb/mlx_sft/`.

In [2]:
# Load SFT trajectories
sft_path = PROJECT_ROOT / "data" / "bbb" / "trajectories_sft.jsonl"
records = []
with open(sft_path) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} trajectories")

# Inspect structure
for i, r in enumerate(records[:3]):
    roles = [m["role"] for m in r["messages"]]
    has_think = any("<think>" in (m.get("content", "") or "") for m in r["messages"])
    print(f"  [{i}] {r.get('ticker','?')} — {len(r['messages'])} msgs, {len(r['tools'])} tools, think={has_think}")
    print(f"       roles: {roles}")

Loaded 10 trajectories
  [0] WDAY — 9 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [1] DUK — 10 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [2] LLY — 9 msgs, 4 tools, think=True
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']


In [3]:
# Split and save for MLX
random.seed(42)
random.shuffle(records)


def to_mlx_format(record: dict) -> dict:
    """
    Convert SFT record to MLX-compatible format.

    Key fix: Qwen3.5's Jinja template uses `arguments | items` which expects a dict,
    but responses_to_hermes() stores arguments as a JSON string. Parse it here.
    """
    messages = []
    for m in record["messages"]:
        m = dict(m)  # shallow copy
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {
                    **tc,
                    "function": {
                        **tc["function"],
                        "arguments": (
                            json.loads(tc["function"]["arguments"])
                            if isinstance(tc["function"]["arguments"], str)
                            else tc["function"]["arguments"]
                        ),
                    },
                }
                for tc in m["tool_calls"]
            ]
        messages.append(m)
    return {"messages": messages, "tools": record["tools"]}


split_idx = int(len(records) * 0.85)
train_records = records[:split_idx]
eval_records = records[split_idx:]

# Ensure at least 1 eval sample
if not eval_records:
    eval_records = [train_records.pop()]

mlx_data_dir = PROJECT_ROOT / "data" / "bbb" / "mlx_sft"
mlx_data_dir.mkdir(parents=True, exist_ok=True)

for split_name, split_data in [("train", train_records), ("valid", eval_records)]:
    path = mlx_data_dir / f"{split_name}.jsonl"
    with open(path, "w") as f:
        for r in split_data:
            f.write(json.dumps(to_mlx_format(r)) + "\n")
    print(f"Saved {len(split_data)} records to {path}")

train_tickers = {r.get("ticker") for r in train_records}
print(f"\nTrain tickers: {train_tickers}")

Saved 8 records to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft/train.jsonl
Saved 2 records to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft/valid.jsonl

Train tickers: {'HSY', 'ECL', 'NKE', 'BABA', 'AMD', 'BA', 'LLY', 'DHR'}


### Training Config

Generate the YAML config for `mlx_lm.lora`. Key settings:
- `mask_prompt: true` — only train on assistant turns (same as Unsloth's `train_on_responses_only`)
- `grad_checkpoint: true` — trades ~30% speed for lower memory (required on 16GB Mac)
- `rank: 32, scale: 2.0` — equivalent to Unsloth's `r=32, lora_alpha=64`
- `learning_rate: 1e-5` — MLX default, lower than Unsloth's 2e-4 (MLX uses full-precision Adam)

In [7]:
import yaml

config = {
    # Model
    "model": "mlx-community/Qwen3.5-4B-MLX-4bit",
    "train": True,
    "data": str(mlx_data_dir),
    # Training
    "iters": 200,
    "batch_size": 1,
    "grad_accumulation_steps": 8,
    "max_seq_length": 16384,
    "mask_prompt": True,
    "grad_checkpoint": True,
    "learning_rate": 1e-5,
    # LoRA
    "lora_parameters": {
        "rank": 16,
        "scale": 2.0,       # = alpha / rank = 64 / 32
        "dropout": 0.0,
    },
    "num_layers": 8,        # all layers would be -1
    # Checkpoints & logging
    "adapter_path": str(PROJECT_ROOT / "models" / "mlx_sft_adapters"),
    "save_every": 50,
    "steps_per_report": 10,
    "steps_per_eval": 50,
}

config_path = PROJECT_ROOT / "nb" / "bbb" / "sft_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to {config_path}")
print(f"\n{yaml.dump(config, default_flow_style=False, sort_keys=False)}")

Config saved to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/nb/bbb/sft_config.yaml

model: mlx-community/Qwen3.5-4B-MLX-4bit
train: true
data: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/mlx_sft
iters: 200
batch_size: 1
grad_accumulation_steps: 8
max_seq_length: 16384
mask_prompt: true
grad_checkpoint: true
learning_rate: 1.0e-05
lora_parameters:
  rank: 16
  scale: 2.0
  dropout: 0.0
num_layers: 8
adapter_path: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/models/mlx_sft_adapters
save_every: 50
steps_per_report: 10
steps_per_eval: 50



### Train

Run `mlx_lm.lora` with the config. This can be run from the cell below or in a separate terminal:

```bash
uv run mlx_lm.lora --config nb/bbb/sft_config.yaml
```

On 16GB M1 with `batch_size=1, grad_checkpoint=true, max_seq_length=4096`, expect ~8-10GB memory
usage and ~1-2 iterations/second.

In [8]:
#!uv run mlx_lm.lora --config {config_path}

### Fuse Adapters

Merge the LoRA weights into the base model to create a standalone model directory.
This fused model can be served directly via `mlx_lm.server`.

In [ ]:
adapter_path = PROJECT_ROOT / "models" / "mlx_sft_adapters"
fused_path = PROJECT_ROOT / "models" / "mlx_sft_fused"

!uv run mlx_lm.fuse \
    --model mlx-community/Qwen3.5-4B-MLX-4bit \
    --adapter-path {adapter_path} \
    --save-path {fused_path}

print(f"Fused model saved to {fused_path}")

### Evaluate SFT Model

Serve the fused model and run the same agent loop from Phase 3 on unseen tickers.

**Start the server in a terminal:**
```bash
uv run mlx_lm.server --model models/mlx_sft_fused --port 8080 --prompt-cache-size 4
```

Then run the cells below.

In [ ]:
from openai import AsyncOpenAI

sft_client = AsyncOpenAI(base_url="http://localhost:8080/v1", api_key="none")

# Warm up cache
warmup = await sft_client.chat.completions.create(
    model="default_model",
    messages=[{"role": "user", "content": "Say hello."}],
    max_tokens=10,
)
print(f"Cache warmed: {warmup.choices[0].message.content!r}")

In [ ]:
# Evaluate on unseen tickers
unseen_tickers = [t for t in TICKERS if t not in train_tickers][:10]
print(f"Testing on {len(unseen_tickers)} unseen tickers: {unseen_tickers}")

sft_results = []

for ticker in unseen_tickers:
    focus = random.choice(["financial health", "growth potential", "competitive position"])
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{ticker} — {focus}")
    try:
        res = await run_tool_calling_agent_chat(
            client=sft_client,
            model="default_model",
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools=TOOL_SCHEMAS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=5,
            max_tokens=4096,
            verbose=True,
        )

        n_tool_calls = sum(len(m.get("tool_calls", [])) for m in res["input"])
        reward = compute_reward(res["input"], VALID_TOOL_NAMES, res.get("reasoning_summaries"))

        sft_results.append({
            "ticker": ticker,
            "focus": focus,
            "n_tool_calls": n_tool_calls,
            "output_len": len(res["output_text"]),
            "reward": reward,
        })
        print(f"  → reward={reward['total']}, tools={n_tool_calls}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        sft_results.append({"ticker": ticker, "focus": focus, "reward": {"total": -3.0}})

In [ ]:
# Compare SFT vs baseline
sft_rewards = [r["reward"]["total"] for r in sft_results if isinstance(r["reward"], dict)]
sft_successful = [r for r in sft_results if isinstance(r["reward"], dict) and "total" in r["reward"]]

print(f"=== SFT MODEL ({len(sft_successful)}/{len(sft_results)} successful) ===")
print(f"Reward:       avg={sum(sft_rewards)/len(sft_rewards):.2f}  min={min(sft_rewards):.2f}  max={max(sft_rewards):.2f}")

components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
print("\n--- Per-component averages ---")
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in sft_successful]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

# Load baseline for comparison
baseline_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results_mlx.jsonl"
if baseline_path.exists():
    baseline_rewards = []
    with open(baseline_path) as f:
        for line in f:
            r = json.loads(line)
            if isinstance(r.get("reward"), dict):
                baseline_rewards.append(r["reward"]["total"])

    print(f"\n{'Metric':<20} {'Baseline':>10} {'SFT':>10} {'Delta':>10}")
    print("-" * 55)
    b_avg = sum(baseline_rewards) / len(baseline_rewards)
    s_avg = sum(sft_rewards) / len(sft_rewards)
    print(f"{'Avg Reward':<20} {b_avg:>+10.2f} {s_avg:>+10.2f} {s_avg - b_avg:>+10.2f}")
else:
    print("\nNo baseline results found — run Phase 3 batch first for comparison")

## Databricks SFT (GPU / Unsloth)

Everything below runs on Databricks with GPU. Uses Unsloth + SFTTrainer with LoRA.
Skip this section if training locally with MLX above.

In [ ]:
%pip install -q unsloth trl datasets

In [ ]:
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import mlflow
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS, TOOL_SCHEMAS
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    _responses_tool_to_chat,
    make_user_prompt,
    trajectory_token_stats,
)
from bbb.helpers__inference import (
    run_local_agent_loop,
    compute_reward,
)

# Derive valid tool names from the tool registry
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

# MLflow experiment — matches winterfest convention
mlflow.set_experiment("/Users/kristof.rabay/qwen3-4b-bbb")

run_name = f"qwen3-4b-sft-bbb-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"
print(run_name)

## Load Model + LoRA

In [ ]:
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Trainable params: {model.print_trainable_parameters()}")

## Load & Format Training Data

Load the SFT trajectories from Phase 2, apply the chat template, and split into train/eval.

In [ ]:
SFT_PATH = PROJECT_ROOT / "data" / "bbb" / "trajectories_sft.jsonl"

sft_records = []
with open(SFT_PATH) as f:
    for line in f:
        sft_records.append(json.loads(line))

print(f"Loaded {len(sft_records)} SFT trajectories")

# Shuffle and split 85/15
random.seed(42)
random.shuffle(sft_records)
split_idx = int(len(sft_records) * 0.85)
train_records = sft_records[:split_idx]
eval_records = sft_records[split_idx:]

print(f"Train: {len(train_records)} | Eval: {len(eval_records)}")

In [ ]:
def format_for_sft(record: dict) -> dict:
    """
    Apply the tokenizer chat template to a Hermes-format trajectory.
    Returns {"text": formatted_string} for SFTTrainer.
    """
    text = tokenizer.apply_chat_template(
        record["messages"],
        tools=record["tools"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


# Format and create HF datasets
train_formatted = [format_for_sft(r) for r in train_records]
eval_formatted = [format_for_sft(r) for r in eval_records]

# Filter out any that are too long
train_formatted = [r for r in train_formatted if r["text"] is not None]
eval_formatted = [r for r in eval_formatted if r["text"] is not None]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Eval dataset: {len(eval_dataset)} examples")

In [ ]:
# Inspect a formatted example
sample = train_formatted[0]["text"]
print(f"Token count: {len(tokenizer.encode(sample))}")
print(f"\nFirst 1500 chars:\n{'='*60}")
print(sample[:1500])
print(f"\n...\n\nLast 500 chars:\n{'='*60}")
print(sample[-500:])

In [ ]:
# Token length distribution
lengths = [len(tokenizer.encode(r["text"])) for r in train_formatted]

print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
over_limit = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"Over {MAX_SEQ_LENGTH}: {over_limit}/{len(lengths)}")
if over_limit:
    print("(These will be clipped during training)")

## Training

SFT with `train_on_responses_only` — only assistant turns contribute to the loss.
System, user, and tool messages are masked (labels=-100).

**Note:** If `train_on_responses_only` raises a ZeroDivisionError with Qwen3
(known issue #2771), see the fallback cell below.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,  # preserve message boundaries for masking

        max_steps=500,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,

        eval_strategy="steps",
        eval_steps=25,
        save_steps=150,
        save_total_limit=3,

        optim="adamw_8bit",
        weight_decay=0.01,

        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        logging_steps=5,

        seed=3407,
        output_dir="outputs",
        report_to="mlflow",
        run_name=run_name,
    ),
)

# Mask non-assistant turns — only train on model outputs
# Qwen3 uses <|im_start|> delimiters
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Verify masking — check that labels are -100 for non-assistant tokens
sample_batch = trainer.get_train_dataloader()
batch = next(iter(sample_batch))

total_tokens = (batch["labels"] != -100).sum().item()
masked_tokens = (batch["labels"] == -100).sum().item()

print(f"Training tokens: {total_tokens}")
print(f"Masked tokens:   {masked_tokens}")
print(f"Train ratio:     {total_tokens / (total_tokens + masked_tokens) * 100:.1f}%")

In [ ]:
trainer_stats = trainer.train()

## Training Analysis

In [ ]:
import pandas as pd

# Extract training history
df_history = pd.DataFrame(trainer.state.log_history)

train_df = df_history[df_history["loss"].notna()]
eval_df = df_history[df_history["eval_loss"].notna()]

if not eval_df.empty:
    final_train = train_df["loss"].iloc[-5:].mean()
    final_eval = eval_df["eval_loss"].iloc[-3:].mean()
    gap = final_eval - final_train

    print(f"Final train loss (avg last 5): {final_train:.4f}")
    print(f"Final eval loss (avg last 3):  {final_eval:.4f}")
    print(f"Gap: {gap:.4f}")
    if gap > 0.1:
        print("Warning: possible overfitting")
    else:
        print("Good generalization")
else:
    print(f"Final train loss: {train_df['loss'].iloc[-1]:.4f}")

In [ ]:
# Plot loss curves
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_df["step"], train_df["loss"], label="Train", alpha=0.7)
    if not eval_df.empty:
        ax.plot(eval_df["step"], eval_df["eval_loss"], label="Eval", marker="o")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("SFT Training — Tool-Calling Qwen3-4B")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping plot")

## Inference Test

Run the fine-tuned model on unseen tickers and compare with baseline.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

from bbb.tools import TOOL_SCHEMAS

TOOLS_CHAT = [_responses_tool_to_chat(t) for t in TOOL_SCHEMAS]

In [ ]:
# Pick tickers NOT in the training set
train_tickers = {r["ticker"] for r in train_records}
unseen_tickers = [t for t in TICKERS if t not in train_tickers][:10]
print(f"Testing on {len(unseen_tickers)} unseen tickers: {unseen_tickers}")

In [ ]:
sft_results = []

for ticker in unseen_tickers:
    focus = random.choice(["financial health", "growth potential", "competitive position"])
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{ticker} — {focus}")

    try:
        res = run_local_agent_loop(
            model=model,
            tokenizer=tokenizer,
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools_chat=TOOLS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=8,
            enable_thinking=True,
            verbose=True,
        )

        reward = compute_reward(res["messages"], VALID_TOOL_NAMES)
        sft_results.append({
            "ticker": ticker,
            "n_tool_calls": res["n_tool_calls"],
            "output_len": len(res["output_text"]),
            "reward": reward,
        })
        print(f"  → reward={reward['total']}, tools={res['n_tool_calls']}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        sft_results.append({"ticker": ticker, "reward": {"total": -3.0}})

# Summary
sft_rewards = [r["reward"]["total"] for r in sft_results]
print(f"\n=== SFT MODEL ===")
print(f"Avg reward: {sum(sft_rewards)/len(sft_rewards):.2f}")
print(f"Completion rate: {sum(1 for r in sft_results if r['reward'].get('completion')) / len(sft_results) * 100:.0f}%")

In [ ]:
# Compare with baseline (load if available)
baseline_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results.jsonl"
if baseline_path.exists():
    baseline_rewards = []
    with open(baseline_path) as f:
        for line in f:
            r = json.loads(line)
            baseline_rewards.append(r["reward"]["total"])

    print(f"{'Metric':<20} {'Baseline':>10} {'SFT':>10}")
    print("-" * 42)
    print(f"{'Avg Reward':<20} {sum(baseline_rewards)/len(baseline_rewards):>+10.2f} {sum(sft_rewards)/len(sft_rewards):>+10.2f}")
else:
    print("No baseline results found — run Phase 3 first for comparison")

## Save Model

Save LoRA adapters for later use (RL, inference, deployment).

In [ ]:
save_dir = PROJECT_ROOT / "models" / "qwen3-4b-tool-calling-sft"
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

print(f"Saved LoRA adapters to {save_dir}")

In [ ]:
# Optional: save as GGUF for llama.cpp inference
# Uncomment to export (takes a few minutes)

# gguf_dir = PROJECT_ROOT / "models" / "qwen3-4b-tool-calling-sft-gguf"
# model.save_pretrained_gguf(
#     str(gguf_dir),
#     tokenizer,
#     quantization_method="Q8_0",
# )
# print(f"Saved GGUF to {gguf_dir}")